In [1]:
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from agent_lab.model_hub import LLM_FAST, LLM_THINKING


chat_model = init_chat_model(**LLM_FAST)
reason_model = init_chat_model(**LLM_THINKING)


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """choose model based on key word."""
    messages = request.state['messages']
    model = chat_model
    for msg in messages:
        if isinstance(msg, HumanMessage) and 'think' in msg.content:
            model = reason_model
    return handler(request.override(model=model))


agent = create_agent(model=chat_model, middleware=[dynamic_model_selection])

In [2]:
deep_response = agent.invoke(
    {
        'messages': [
            HumanMessage('Please derive the root formula for the cubic equation, think deeply.')
        ]
    }
)

normal_response = agent.invoke(
    {
        'messages': [
            HumanMessage('What is the weather like today?')
        ]
    }
)

In [3]:
deep_msg = deep_response['messages'][-1]
deep_msg.response_metadata

{'token_usage': {'completion_tokens': 3006,
  'prompt_tokens': 17,
  'total_tokens': 3023,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 1304,
   'rejected_prediction_tokens': None},
  'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
  'prompt_cache_hit_tokens': 0,
  'prompt_cache_miss_tokens': 17},
 'model_provider': 'deepseek',
 'model_name': 'deepseek-v4-flash',
 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402',
 'id': '059f3897-572d-480f-abd3-508cbf27f412',
 'finish_reason': 'stop',
 'logprobs': None}

In [4]:
deep_msg.additional_kwargs

{'refusal': None,
 'reasoning_content': 'We need to derive the root formula for the cubic equation. The cubic equation is typically written as: ax^3 + bx^2 + cx + d = 0, with a ≠ 0. The general solution is known as Cardano\'s formula. We\'ll derive it step by step.\n\nFirst, we can reduce the general cubic to a depressed cubic (without quadratic term) by substituting x = y - b/(3a). This is a standard technique. Let\'s do that.\n\nGiven: ax^3 + bx^2 + cx + d = 0. Divide by a: x^3 + (b/a)x^2 + (c/a)x + (d/a) = 0. Let p = b/a, q = c/a, r = d/a. Then x^3 + p x^2 + q x + r = 0.\n\nNow substitute x = y - p/3. Then expand: (y - p/3)^3 + p(y - p/3)^2 + q(y - p/3) + r = 0.\n\nCompute: (y^3 - p y^2 + (p^2/3) y - p^3/27) + p(y^2 - (2p/3)y + p^2/9) + q y - q p/3 + r = 0.\n\nCombine terms: y^3 term: y^3. y^2 terms: -p y^2 + p y^2 = 0. Good. y terms: (p^2/3)y - (2p^2/3)y + q y = (p^2/3 - 2p^2/3 + q) y = (-p^2/3 + q) y. Constant terms: -p^3/27 + p*(p^2/9) - q p/3 + r = -p^3/27 + p^3/9 - (q p)/3 + r 

In [5]:
deep_msg.pretty_print()

================================== Ai Message ==================================

The general cubic equation is \(ax^3 + bx^2 + cx + d = 0\) with \(a \neq 0\). We derive the formula for its roots by reducing it to a depressed cubic (without the quadratic term) and then solving using the substitution \(y = u + v\).

### Step 1: Depress the cubic
Divide by \(a\):
\[
x^3 + \frac{b}{a}x^2 + \frac{c}{a}x + \frac{d}{a} = 0.
\]
Define \(p = \frac{b}{a}\), \(q = \frac{c}{a}\), \(r = \frac{d}{a}\); then
\[
x^3 + p x^2 + q x + r = 0.
\]
Let \(x = y - \frac{p}{3}\). Substituting and expanding:
\[
\left(y - \frac{p}{3}\right)^3 + p\left(y - \frac{p}{3}\right)^2 + q\left(y - \frac{p}{3}\right) + r = 0.
\]
Compute:
\[
\begin{aligned}
\left(y - \frac{p}{3}\right)^3 &= y^3 - p y^2 + \frac{p^2}{3}y - \frac{p^3}{27}, \\
p\left(y - \frac{p}{3}\right)^2 &= p\left(y^2 - \frac{2p}{3}y + \frac{p^2}{9}\right) = p y^2 - \frac{2p^2}{3}y + \frac{p^3}{9}, \\
q\left(y - \frac{p}{3}\right) &= q y - \frac{q p}{3}.
\

In [6]:
normal_msg = normal_response['messages'][-1]
normal_msg.response_metadata

{'token_usage': {'completion_tokens': 78,
  'prompt_tokens': 11,
  'total_tokens': 89,
  'completion_tokens_details': None,
  'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
  'prompt_cache_hit_tokens': 0,
  'prompt_cache_miss_tokens': 11},
 'model_provider': 'deepseek',
 'model_name': 'deepseek-v4-flash',
 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402',
 'id': '46dbd2ae-8fbd-4b81-9091-54f431456431',
 'finish_reason': 'stop',
 'logprobs': None}

In [7]:
normal_msg.additional_kwargs

{'refusal': None}

In [8]:
normal_msg.pretty_print()

================================== Ai Message ==================================

I don't have access to your current location or real-time weather data, so I can't tell you the weather where you are right now. However, you can check a weather website or app (like Weather.com or AccuWeather) for your local forecast, or simply look outside! 😊 If you tell me your location, I can help you find resources to check the weather.
